# 04: Run your circuit on a remote CPU

**What you will learn:** a cloud job runs elsewhere and has its own ID, status and result. The circuit is still the same FlagQuantum circuit.

We will send one Bell circuit to Jiuding. This is a CPU experiment, not an accelerator or speedup demonstration.


## 1. Prepare the workshop

Run this setup cell unchanged. It finds your personal result directory. The event environment supplies credentials and the job image; no secrets belong in notebook cells. See [Setup](../../SETUP.md) when using your own computer.


In [ ]:
from pathlib import Path
import sys

search_roots = (Path.cwd().resolve(), *Path.cwd().resolve().parents)
WORKSHOP = next((root / "flagos2026" for root in search_roots
                 if (root / "flagos2026/notebooks").is_dir()), None)
if WORKSHOP is None:
    raise RuntimeError("Cannot locate flagos2026/notebooks. Open the deployed workshop first.")
sys.path.insert(0, str(WORKSHOP / "scripts"))
from cloud_lesson import prepare_lesson

lesson = prepare_lesson(Path.cwd(), "jiuding-bell-lesson")
WORKSHOP = lesson.workshop
repository = WORKSHOP.parent.parent
ROOT = repository if (repository / "pyproject.toml").is_file() else WORKSHOP
OUTPUTS = lesson.outputs


## 2. Build a Bell circuit and predict the answer

The ideal probabilities for `00` and `11` are each one half. First check this locally.


In [ ]:
import flagquantum as fq

circuit = fq.Circuit(2).h(0).cx(0, 1)
local = fq.run(circuit)
print("Local probabilities:", local.state.abs().square().tolist())


## 3. Submit one real cloud job

Set the switch to `True` when you are ready. **Running this cell then creates a real CPU task and consumes queue resources.**

`target` chooses where to run; `image` supplies its installed software. The workshop shortcut below calls `fq.submit()` and automatically saves a receipt. Rerunning restores the same job, rather than creating a duplicate.


In [ ]:
target = "jiuding:cpu"
submit_to_cloud = False

job = lesson.submit_once(
    circuit, target=target, image=lesson.image, enabled=submit_to_cloud,
)
if job is not None:
    print("Job ID:", job.id)


## 4. Inspect its status

`queued` means waiting for resources; `running` means executing; `succeeded` means a result is available. This cell only queries the saved job.


In [ ]:
if job is not None:
    print("Status:", job.status())


## 5. Compare the remote and local answers

The result cell waits briefly. If the queue is busy, rerun **this cell** later—do not create another task. A waiting task is not a failed calculation.


In [ ]:
remote = lesson.collect(job)
if remote is not None:
    import torch

    torch.testing.assert_close(remote.state, local.state)
    print("Remote probabilities:", remote.state.abs().square().tolist())
    print("The remote CPU calculation matches the local calculation.")


## Explain what happened

- Which cell actually submits work, and which cells only inspect it?
- Why does a task ID not yet prove the calculation has finished?
- Does running this small circuit remotely prove a performance improvement? Why not?

**Try next:** change the circuit and choose a new lesson name in setup so the experiment keeps a separate receipt.

For complete Python programs, accelerator selection and recovery details, use [Cloud job field notes](../../CLOUD_JOBS.md). They are separate from this circuit lesson.
